# 02 - ResNet18 - ImageNet

Generated for Colab. The repo URL is set to `https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git`.


## Before running
These notebooks are designed for **Colab**.

The real processed data is expected inside the cloned repo:

```text
/content/contrastive-synthesis-medcls_CVProject/data/processed/
├── labelled_4232/
│   ├── COVID/images/
│   ├── Lung_Opacity/images/
│   ├── Viral_Pneumonia/images/
│   └── Normal/images/
└── unlabelled_16934/images/
```

Notebook `00_train_compare_gans_acgan_dcgan.ipynb` saves generated synthetic images to **Google Drive** so they persist after Colab disconnects:

```text
/content/drive/MyDrive/medcls_cvproject/data/processed/synthetic_dcgan/
/content/drive/MyDrive/medcls_cvproject/data/processed/synthetic_acgan/
```

The synthetic classification notebooks (`05`, `06`, `11`, `12`) load `synthetic_dcgan` from Google Drive, so run notebook `00` first.


In [ ]:

# =========================
# 1. Colab / Drive / Repo setup
# =========================
import os, sys, json, math, random, time, copy, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

REPO_URL = "https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git"
REPO_DIR = Path('/content/contrastive-synthesis-medcls_CVProject')
DRIVE_ROOT = Path('/content/drive/MyDrive/medcls_cvproject')
REPO_DATA_ROOT = REPO_DIR / 'data' / 'processed'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data' / 'processed'
LABELLED_DATA = REPO_DATA_ROOT / 'labelled_4232'
UNLABELLED_REAL_DATA = REPO_DATA_ROOT / 'unlabelled_16934' / 'images'
SYNTHETIC_DCGAN_DATA = DRIVE_DATA_ROOT / 'synthetic_dcgan'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'classification'

EXPERIMENT_NAME = "02_resnet18_imagenet"
BACKBONE_NAME = "resnet18"       # resnet18 or vit_s16
STRATEGY = "ImageNet"                 # None, ImageNet, COVID-QU, ImageNet_to_COVID-QU, COVID-QU-Syn, ImageNet_to_COVID-QU-Syn

# Keep this True for the real report-style run. Set False for a quick smoke test.
FULL_RUN = True
FORCE_RETRAIN_PRETRAIN = False
FORCE_RETRAIN_FINETUNE = False
SEED = 42
CLASSES = ['COVID', 'Lung_Opacity', 'Viral_Pneumonia', 'Normal']

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Experiment:', EXPERIMENT_NAME)
print('Backbone:', BACKBONE_NAME)
print('Strategy:', STRATEGY)
print('Repo labelled data:', LABELLED_DATA)
print('Repo unlabelled real data:', UNLABELLED_REAL_DATA)
print('Drive synthetic DCGAN data:', SYNTHETIC_DCGAN_DATA)
print('Output:', OUTPUT_DIR)

# Clone the repo mainly to keep the run tied to your project repository.
# The training code below is self-contained so the notebook still shows useful errors even if clone fails.
if not REPO_DIR.exists():
    print('Cloning repo...')
    result = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print('WARNING: repo clone failed. Check repo URL / visibility. Error:')
        print(result.stderr)
else:
    print('Repo already exists:', REPO_DIR)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    print('Working directory:', Path.cwd())

# Install dependencies.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
if (REPO_DIR / 'requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn', 'scipy', 'pandas', 'tqdm', 'seaborn', 'matplotlib'])


In [ ]:

# =========================
# 2. Imports and utilities
# =========================
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import timm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU, running on CPU')
print('Device:', DEVICE)

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXTS and p.is_file()]

def count_images(root):
    return len(list_images(root))

print('Labelled images:', count_images(LABELLED_DATA))
print('Unlabelled real images:', count_images(UNLABELLED_REAL_DATA))
print('Synthetic DCGAN images:', count_images(SYNTHETIC_DCGAN_DATA))

assert LABELLED_DATA.exists(), f'Missing labelled data: {LABELLED_DATA}'
if STRATEGY in ['COVID-QU', 'ImageNet_to_COVID-QU']:
    assert count_images(UNLABELLED_REAL_DATA) > 0, f'Missing real unlabelled data: {UNLABELLED_REAL_DATA}'
if STRATEGY in ['COVID-QU-Syn', 'ImageNet_to_COVID-QU-Syn']:
    assert count_images(SYNTHETIC_DCGAN_DATA) > 0, f'Missing synthetic DCGAN data. Run the GAN notebook first: {SYNTHETIC_DCGAN_DATA}'


In [ ]:

# =========================
# 3. Dataset definitions
# =========================
class LabelledImageDataset(Dataset):
    def __init__(self, root, classes, transform=None, max_per_class=None):
        self.root = Path(root)
        self.classes = classes
        self.transform = transform
        self.samples = []
        for label, cls in enumerate(classes):
            class_root = self.root / cls
            img_root = class_root / 'images' if (class_root / 'images').exists() else class_root
            files = list_images(img_root)
            files = sorted(files)
            if max_per_class is not None:
                files = files[:max_per_class]
            self.samples.extend([(p, label) for p in files])
        if not self.samples:
            raise ValueError(f'No labelled images found in {root}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

class UnlabelledImageDataset(Dataset):
    def __init__(self, root, transform=None, max_images=None):
        self.root = Path(root)
        self.transform = transform
        self.files = sorted(list_images(self.root))
        if max_images is not None:
            self.files = self.files[:max_images]
        if not self.files:
            raise ValueError(f'No unlabelled images found in {root}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform:
            return self.transform(img)
        return img

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform
    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

ssl_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    normalize,
])

MAX_PER_CLASS = None if FULL_RUN else 80
MAX_SSL_IMAGES = None if FULL_RUN else 512
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Report-like settings.
PRETRAIN_EPOCHS = 70 if BACKBONE_NAME == 'resnet18' else 120
FINETUNE_EPOCHS = 70 if BACKBONE_NAME == 'resnet18' else 50
PRETRAIN_BATCH_SIZE = 64
FINETUNE_BATCH_SIZE = 32

if not FULL_RUN:
    PRETRAIN_EPOCHS = 1
    FINETUNE_EPOCHS = 1
    PRETRAIN_BATCH_SIZE = 16
    FINETUNE_BATCH_SIZE = 16

print('PRETRAIN_EPOCHS:', PRETRAIN_EPOCHS)
print('FINETUNE_EPOCHS:', FINETUNE_EPOCHS)
print('NUM_WORKERS:', NUM_WORKERS)
print('PIN_MEMORY:', PIN_MEMORY)


In [ ]:

# =========================
# 4. Model builders
# =========================
def build_feature_backbone(backbone_name, pretrained=False):
    if backbone_name == 'resnet18':
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = resnet18(weights=weights)
        model.fc = nn.Identity()
        feat_dim = 512
        return model, feat_dim
    elif backbone_name == 'vit_s16':
        model = timm.create_model('vit_small_patch16_224', pretrained=pretrained, num_classes=0)
        feat_dim = model.num_features
        return model, feat_dim
    else:
        raise ValueError(backbone_name)

class FineTuneModel(nn.Module):
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        return self.head(self.backbone(x))

def load_backbone_state(backbone, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('backbone_state', ckpt)
    missing, unexpected = backbone.load_state_dict(state, strict=False)
    print('Loaded backbone checkpoint:', ckpt_path)
    print('Missing keys:', len(missing), 'Unexpected keys:', len(unexpected))
    return backbone


In [ ]:

# =========================
# 5. Self-supervised pretraining: SimCLR for ResNet, DINO for ViT
# =========================
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    def __init__(self, backbone, feat_dim, out_dim=128):
        super().__init__()
        self.backbone = backbone
        self.projector = ProjectionHead(feat_dim, 512, out_dim)
    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)

def nt_xent_loss(z1, z2, temperature=0.5):
    bsz = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2 * bsz, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)
    pos = torch.cat([torch.diag(sim, bsz), torch.diag(sim, -bsz)], dim=0)
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()

class DINOHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, bottleneck_dim=256, out_dim=4096):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim), nn.GELU(),
        )
        self.last = nn.Linear(bottleneck_dim, out_dim)
    def forward(self, x):
        return self.last(self.mlp(x))

def dino_loss(student_out, teacher_out, center, student_temp=0.1, teacher_temp=0.04):
    student_logp = F.log_softmax(student_out / student_temp, dim=-1)
    teacher_p = F.softmax((teacher_out - center) / teacher_temp, dim=-1).detach()
    return torch.sum(-teacher_p * student_logp, dim=-1).mean()

def train_simclr_pretrain(data_root, output_path, start_from_imagenet=False):
    output_path = Path(output_path)
    if output_path.exists() and not FORCE_RETRAIN_PRETRAIN:
        print('Using existing SimCLR checkpoint:', output_path)
        return output_path

    dataset = UnlabelledImageDataset(data_root, transform=TwoCropsTransform(ssl_tf), max_images=MAX_SSL_IMAGES)
    loader = DataLoader(dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY)

    backbone, feat_dim = build_feature_backbone('resnet18', pretrained=start_from_imagenet)
    model = SimCLR(backbone, feat_dim).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(PRETRAIN_EPOCHS):
        model.train()
        losses = []
        for x1, x2 in tqdm(loader, desc=f'SimCLR epoch {epoch+1}/{PRETRAIN_EPOCHS}'):
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1, z2 = model(x1), model(x2)
            loss = nt_xent_loss(z1, z2, temperature=0.5)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f'Epoch {epoch+1}: SimCLR loss={np.mean(losses):.4f}')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'backbone_state': model.backbone.state_dict(), 'method': 'SimCLR', 'start_from_imagenet': start_from_imagenet}, output_path)
    print('Saved:', output_path)
    return output_path

def train_dino_pretrain(data_root, output_path, start_from_imagenet=False):
    output_path = Path(output_path)
    if output_path.exists() and not FORCE_RETRAIN_PRETRAIN:
        print('Using existing DINO checkpoint:', output_path)
        return output_path

    dataset = UnlabelledImageDataset(data_root, transform=TwoCropsTransform(ssl_tf), max_images=MAX_SSL_IMAGES)
    loader = DataLoader(dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY)

    student, feat_dim = build_feature_backbone('vit_s16', pretrained=start_from_imagenet)
    teacher = copy.deepcopy(student)
    for p in teacher.parameters():
        p.requires_grad = False

    student_head = DINOHead(feat_dim).to(DEVICE)
    teacher_head = copy.deepcopy(student_head).to(DEVICE)
    student = student.to(DEVICE)
    teacher = teacher.to(DEVICE)
    for p in teacher_head.parameters():
        p.requires_grad = False

    params = list(student.parameters()) + list(student_head.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.04)
    center = torch.zeros(1, 4096, device=DEVICE)
    center_momentum = 0.9
    teacher_momentum = 0.996

    for epoch in range(PRETRAIN_EPOCHS):
        student.train(); student_head.train()
        teacher.eval(); teacher_head.eval()
        losses = []
        for x1, x2 in tqdm(loader, desc=f'DINO epoch {epoch+1}/{PRETRAIN_EPOCHS}'):
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            s1 = student_head(student(x1))
            s2 = student_head(student(x2))
            with torch.no_grad():
                t1 = teacher_head(teacher(x1))
                t2 = teacher_head(teacher(x2))
            loss = 0.5 * (dino_loss(s1, t2, center) + dino_loss(s2, t1, center))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                batch_center = torch.cat([t1, t2], dim=0).mean(dim=0, keepdim=True)
                center = center * center_momentum + batch_center * (1 - center_momentum)
                for ps, pt in zip(student.parameters(), teacher.parameters()):
                    pt.data.mul_(teacher_momentum).add_(ps.data, alpha=1 - teacher_momentum)
                for ps, pt in zip(student_head.parameters(), teacher_head.parameters()):
                    pt.data.mul_(teacher_momentum).add_(ps.data, alpha=1 - teacher_momentum)
            losses.append(loss.item())
        print(f'Epoch {epoch+1}: DINO loss={np.mean(losses):.4f}')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'backbone_state': student.state_dict(), 'method': 'DINO', 'start_from_imagenet': start_from_imagenet}, output_path)
    print('Saved:', output_path)
    return output_path

def maybe_run_pretraining():
    if STRATEGY in ['None', 'ImageNet']:
        return None
    if STRATEGY in ['COVID-QU', 'ImageNet_to_COVID-QU']:
        data_root = UNLABELLED_REAL_DATA
    elif STRATEGY in ['COVID-QU-Syn', 'ImageNet_to_COVID-QU-Syn']:
        data_root = SYNTHETIC_DCGAN_DATA
    else:
        raise ValueError(STRATEGY)

    start_imgnet = STRATEGY.startswith('ImageNet_to')
    ckpt_name = f'{EXPERIMENT_NAME}_ssl_backbone.pt'
    ckpt_path = OUTPUT_DIR / 'pretrain' / ckpt_name

    if BACKBONE_NAME == 'resnet18':
        return train_simclr_pretrain(data_root, ckpt_path, start_from_imagenet=start_imgnet)
    elif BACKBONE_NAME == 'vit_s16':
        return train_dino_pretrain(data_root, ckpt_path, start_from_imagenet=start_imgnet)
    else:
        raise ValueError(BACKBONE_NAME)


In [ ]:

# =========================
# 6. Full supervised fine-tuning and evaluation
# =========================
def make_splits(n, train_split=0.8, val_split=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_split * n)
    n_val = int(val_split * n)
    return idx[:n_train].tolist(), idx[n_train:n_train+n_val].tolist(), idx[n_train+n_val:].tolist()

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    losses, preds, targets = [], [], []
    for x, y in tqdm(loader, desc='train', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds.extend(out.argmax(1).detach().cpu().numpy())
        targets.extend(y.detach().cpu().numpy())
    return np.mean(losses), accuracy_score(targets, preds)

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    losses, preds, targets = [], [], []
    for x, y in tqdm(loader, desc='eval', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        if criterion is not None:
            losses.append(criterion(out, y).item())
        preds.extend(out.argmax(1).detach().cpu().numpy())
        targets.extend(y.detach().cpu().numpy())
    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average='weighted', zero_division=0)
    f1_macro = precision_recall_fscore_support(targets, preds, average='macro', zero_division=0)[2]
    cm = confusion_matrix(targets, preds, labels=list(range(len(CLASSES))))
    return {
        'loss': float(np.mean(losses)) if losses else None,
        'accuracy': float(acc),
        'precision_weighted': float(precision),
        'recall_weighted': float(recall),
        'f1_weighted': float(f1),
        'f1_macro': float(f1_macro),
        'confusion_matrix': cm,
        'targets': targets,
        'preds': preds,
    }

def build_finetune_model(ssl_ckpt_path=None):
    # ImageNet used directly only for the ImageNet baseline.
    # For ImageNet_to_* runs, ImageNet is already used to initialize the SSL pretraining stage.
    direct_imagenet = STRATEGY == 'ImageNet'
    backbone, feat_dim = build_feature_backbone(BACKBONE_NAME, pretrained=direct_imagenet)
    if ssl_ckpt_path is not None:
        backbone = load_backbone_state(backbone, ssl_ckpt_path)
    model = FineTuneModel(backbone, feat_dim, len(CLASSES))
    return model

def run_finetuning(ssl_ckpt_path=None):
    best_path = OUTPUT_DIR / 'finetune' / 'best_model.pt'
    metrics_path = OUTPUT_DIR / 'finetune' / 'test_metrics.json'
    if best_path.exists() and metrics_path.exists() and not FORCE_RETRAIN_FINETUNE:
        print('Fine-tuned model already exists:', best_path)
        with open(metrics_path, 'r') as f:
            print(json.dumps(json.load(f), indent=2))
        return best_path

    dataset_for_split = LabelledImageDataset(LABELLED_DATA, CLASSES, transform=None, max_per_class=MAX_PER_CLASS)
    train_idx, val_idx, test_idx = make_splits(len(dataset_for_split), seed=SEED)

    train_full = LabelledImageDataset(LABELLED_DATA, CLASSES, transform=train_tf, max_per_class=MAX_PER_CLASS)
    val_full = LabelledImageDataset(LABELLED_DATA, CLASSES, transform=val_tf, max_per_class=MAX_PER_CLASS)

    train_ds = Subset(train_full, train_idx)
    val_ds = Subset(val_full, val_idx)
    test_ds = Subset(val_full, test_idx)

    train_loader = DataLoader(train_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    print(f'Split sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

    model = build_finetune_model(ssl_ckpt_path).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    lr = 1e-5
    if BACKBONE_NAME == 'vit_s16' and STRATEGY == 'None':
        lr = 1e-4
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    history = []
    patience, bad_epochs = 10, 0
    best_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(FINETUNE_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = evaluate(model, val_loader, criterion)
        scheduler.step(val_metrics['loss'])
        row = {
            'epoch': epoch + 1,
            'train_loss': float(train_loss),
            'train_acc': float(train_acc),
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'val_f1_macro': val_metrics['f1_macro'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        print(row)
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            bad_epochs = 0
            torch.save({'model_state': model.state_dict(), 'config': {'backbone': BACKBONE_NAME, 'strategy': STRATEGY, 'classes': CLASSES}}, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print('Early stopping')
                break

    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'finetune' / 'history.csv', index=False)

    # Test evaluation on best checkpoint.
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    test_metrics = evaluate(model, test_loader, criterion)

    report = classification_report(test_metrics['targets'], test_metrics['preds'], target_names=CLASSES, zero_division=0, output_dict=True)
    cm = test_metrics['confusion_matrix']

    serializable = {k: v for k, v in test_metrics.items() if k not in ['confusion_matrix', 'targets', 'preds']}
    serializable['classification_report'] = report
    serializable['experiment_name'] = EXPERIMENT_NAME
    serializable['backbone'] = BACKBONE_NAME
    serializable['strategy'] = STRATEGY

    with open(metrics_path, 'w') as f:
        json.dump(serializable, f, indent=2)

    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUTPUT_DIR / 'finetune' / 'confusion_matrix.csv')
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(EXPERIMENT_NAME)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'finetune' / 'confusion_matrix.png', dpi=160)
    plt.show()

    print('Test metrics:')
    print(json.dumps(serializable, indent=2))
    print('Saved best model:', best_path)
    return best_path

ssl_ckpt = maybe_run_pretraining()
best_model = run_finetuning(ssl_ckpt)
print('DONE:', EXPERIMENT_NAME)
